# Distribution integrity audit
Checks release hashes and parsed row counts. Cleanup definitions and coverage limitations are in README.md. No pattern exploration is performed.

In [ ]:
from pathlib import Path
import csv, json, hashlib
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
manifest_path = root / 'data' / 'metadata' / 'manifest.json' if (root / 'data' / 'metadata' / 'manifest.json').exists() else root / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
for item in manifest:
    path = root / 'data' / 'raw' / item['file']
    if not path.exists(): path = root / item['file']
    if not path.exists():
        print(f"{item['file']} not found, skipping")
        continue
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    assert h.hexdigest() == item['sha256']
    with path.open(newline='') as f: count = sum(1 for _ in csv.DictReader(f))
    assert count == item['rows']
    print(item['file'], count, 'verified')
audit_path = root / 'data' / 'metadata' / 'quality_audit.json' if (root / 'data' / 'metadata' / 'quality_audit.json').exists() else root / 'quality_audit.json'
audit = json.loads(audit_path.read_text())
audit['release_checks']
